# Gemini: `maxItems` Cumulative Budget Causes Opaque `INVALID_ARGUMENT`

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/radishbuild/field-notes/blob/main/gemini-maxitems-cumulative-budget/repro_minimal.ipynb)

## What this reproduces

When using Gemini's function calling, the API returns an opaque `INVALID_ARGUMENT` error when tool schemas contain array properties whose **`maxItems` values sum to ~960 or more** across all properties. The error provides zero diagnostic information.

Unlike the [enum complexity bug](../gemini-any-mode-enum-bug/) which only affects ANY mode, this issue affects **both ANY and AUTO modes**.

This notebook demonstrates that:

1. **It's the cumulative sum that matters** — 1 field with `maxItems=972` fails, but so do 8 fields with `maxItems=120` each (sum=960)
2. **Other numeric keywords are unaffected** — `maximum`, `minimum`, `maxLength` at identical values all pass
3. **Stripping `maxItems` entirely fixes it** — Same schemas without `maxItems` always succeed

## Hypothesis

Gemini imposes an undocumented cumulative budget (~960) on the total sum of `maxItems` values across all properties in a tool's parameter schema. When the sum exceeds this threshold, the request is rejected with a bare `INVALID_ARGUMENT` error. Other JSON Schema numeric constraints (`maximum`, `minimum`, `maxLength`) do not contribute to this budget.

In [4]:
!pip install -q google-genai

## API Key Setup

Go to **Colab menu → 🔑 Secrets** (key icon in left sidebar), add a secret named `GEMINI_API_KEY` with your [Gemini API key](https://aistudio.google.com/apikey), and enable notebook access.

In [ ]:
import os
from google import genai
from google.genai import types

try:
    from google.colab import userdata
    api_key = userdata.get("GEMINI_API_KEY")
except ImportError:
    api_key = os.environ["GEMINI_API_KEY"]


client = genai.Client(api_key=api_key)
print("Client ready ✓")

Client ready ✓


## Configuration

In [6]:
MODEL = "gemini-3-flash-preview"

## Schema Generation

We generate a single tool with N array properties, each having a configurable `maxItems` value. The tool is otherwise minimal — string array items, no enums, no nested objects. This isolates `maxItems` as the only variable.

In [7]:
def make_tool(n_fields, max_items):
    """Generate a single tool with n_fields array properties, each with maxItems=max_items."""
    props = {}
    for i in range(n_fields):
        props[f"field_{i:03d}"] = {
            "type": "array",
            "description": f"Test array field {i}",
            "items": {"type": "string"},
            "maxItems": max_items,
        }
    return {
        "name": "search_leads",
        "description": "Search for leads matching the given filters.",
        "parametersJsonSchema": {
            "type": "object",
            "properties": props,
            "required": [],
        },
    }


def make_tool_with_keyword(n_fields, keyword, value):
    """Generate a tool using a different numeric keyword instead of maxItems."""
    props = {}
    for i in range(n_fields):
        if keyword in ("maximum", "minimum"):
            props[f"field_{i:03d}"] = {
                "type": "number",
                "description": f"Test number field {i}",
                keyword: value,
            }
        elif keyword == "maxLength":
            props[f"field_{i:03d}"] = {
                "type": "string",
                "description": f"Test string field {i}",
                keyword: value,
            }
        else:
            props[f"field_{i:03d}"] = {
                "type": "array",
                "description": f"Test array field {i}",
                "items": {"type": "string"},
                keyword: value,
            }
    return {
        "name": "search_leads",
        "description": "Search for leads matching the given filters.",
        "parametersJsonSchema": {
            "type": "object",
            "properties": props,
            "required": [],
        },
    }


def try_call(client, tool_dict, mode="ANY"):
    """Try a generate_content call. Returns True if successful."""
    try:
        fd = types.FunctionDeclaration(**tool_dict)
        client.models.generate_content(
            model=MODEL,
            contents="Search for software engineers at Google",
            config=types.GenerateContentConfig(
                temperature=0,
                tools=[types.Tool(function_declarations=[fd])],
                tool_config=types.ToolConfig(
                    function_calling_config=types.FunctionCallingConfig(mode=mode),
                ),
            ),
        )
        return True
    except Exception:
        return False


print("Helpers ready ✓")

Helpers ready ✓


## Test 1: Single Field — Find the Boundary

With a single array property, sweep `maxItems` from 960 to 980 to pinpoint where the budget kicks in.

In [9]:
print("1 field × maxItems=N  (sum = N)")
print(f"{'maxItems':>10}  │  {'ANY':>6}  {'AUTO':>6}")
print("─" * 34)

for max_items in range(960, 981, 4):
    tool = make_tool(1, max_items)
    any_ok = try_call(client, tool, "ANY")
    auto_ok = try_call(client, tool, "AUTO")
    any_tag = "PASS ✓" if any_ok else "FAIL ✗"
    auto_tag = "PASS ✓" if auto_ok else "FAIL ✗"
    print(f"{max_items:>10}  │  {any_tag:>6}  {auto_tag:>6}")

1 field × maxItems=N  (sum = N)
  maxItems  │     ANY    AUTO
──────────────────────────────────
       960  │  PASS ✓  PASS ✓
       964  │  PASS ✓  PASS ✓
       968  │  PASS ✓  PASS ✓
       972  │  FAIL ✗  PASS ✓
       976  │  FAIL ✗  PASS ✓
       980  │  FAIL ✗  PASS ✓


## Test 2: Multiple Fields — It's the Sum That Matters

With 8 array properties, sweep `maxItems` per field. The budget is cumulative: 8 × 119 = 952 passes, but 8 × 120 = 960 fails.

In [11]:
print("8 fields × maxItems=N  (sum = 8×N)")
print(f"{'maxItems':>10}  {'sum':>6}  │  {'ANY':>6}  {'AUTO':>6}")
print("─" * 40)

for max_items in range(115, 126, 3):
    total = 8 * max_items
    tool = make_tool(8, max_items)
    any_ok = try_call(client, tool, "ANY")
    auto_ok = try_call(client, tool, "AUTO")
    any_tag = "PASS ✓" if any_ok else "FAIL ✗"
    auto_tag = "PASS ✓" if auto_ok else "FAIL ✗"
    print(f"{max_items:>10}  {total:>6}  │  {any_tag:>6}  {auto_tag:>6}")

8 fields × maxItems=N  (sum = 8×N)
  maxItems     sum  │     ANY    AUTO
────────────────────────────────────────
       115     920  │  PASS ✓  PASS ✓
       118     944  │  PASS ✓  PASS ✓
       121     968  │  FAIL ✗  PASS ✓
       124     992  │  FAIL ✗  PASS ✓


## Test 3: Different Field Counts, Same Sum

Same total `maxItems` sum (~960) distributed across different numbers of fields. If it's truly a cumulative budget, all configurations near the threshold should behave the same regardless of distribution.

In [12]:
print("Different distributions of the same cumulative sum")
print(f"{'Fields':>6} × {'maxItems':>8} = {'Sum':>5}  │  {'ANY':>6}  {'AUTO':>6}")
print("─" * 46)

# Just under budget (~950)
configs_under = [
    (1, 950),
    (2, 475),
    (5, 190),
    (10, 95),
    (20, 47),
    (50, 19),
]

# Over budget (~980)
configs_over = [
    (1, 980),
    (2, 490),
    (5, 196),
    (10, 98),
    (20, 49),
    (50, 20),
]

print("--- Under budget (~950) ---")
for n_fields, max_items in configs_under:
    total = n_fields * max_items
    tool = make_tool(n_fields, max_items)
    any_ok = try_call(client, tool, "ANY")
    auto_ok = try_call(client, tool, "AUTO")
    any_tag = "PASS ✓" if any_ok else "FAIL ✗"
    auto_tag = "PASS ✓" if auto_ok else "FAIL ✗"
    print(f"{n_fields:>6} × {max_items:>8} = {total:>5}  │  {any_tag:>6}  {auto_tag:>6}")

print("\n--- Over budget (~980-1000) ---")
for n_fields, max_items in configs_over:
    total = n_fields * max_items
    tool = make_tool(n_fields, max_items)
    any_ok = try_call(client, tool, "ANY")
    auto_ok = try_call(client, tool, "AUTO")
    any_tag = "PASS ✓" if any_ok else "FAIL ✗"
    auto_tag = "PASS ✓" if auto_ok else "FAIL ✗"
    print(f"{n_fields:>6} × {max_items:>8} = {total:>5}  │  {any_tag:>6}  {auto_tag:>6}")

Different distributions of the same cumulative sum
Fields × maxItems =   Sum  │     ANY    AUTO
──────────────────────────────────────────────
--- Under budget (~950) ---
     1 ×      950 =   950  │  PASS ✓  PASS ✓
     2 ×      475 =   950  │  PASS ✓  PASS ✓
     5 ×      190 =   950  │  PASS ✓  PASS ✓
    10 ×       95 =   950  │  PASS ✓  PASS ✓
    20 ×       47 =   940  │  FAIL ✗  PASS ✓
    50 ×       19 =   950  │  FAIL ✗  PASS ✓

--- Over budget (~980-1000) ---
     1 ×      980 =   980  │  FAIL ✗  PASS ✓
     2 ×      490 =   980  │  FAIL ✗  PASS ✓
     5 ×      196 =   980  │  FAIL ✗  PASS ✓
    10 ×       98 =   980  │  FAIL ✗  PASS ✓
    20 ×       49 =   980  │  FAIL ✗  PASS ✓
    50 ×       20 =  1000  │  FAIL ✗  PASS ✓


## Test 4: Control — Stripping `maxItems` Fixes It

Same schemas that failed above, but with `maxItems` removed. All should pass, proving the issue is `maxItems`-specific.

In [13]:
def strip_max_items(tool_dict):
    """Remove maxItems from all properties in a tool schema."""
    import copy
    tool = copy.deepcopy(tool_dict)
    for prop in tool["parametersJsonSchema"]["properties"].values():
        prop.pop("maxItems", None)
    return tool


print("Same failing schemas with maxItems stripped")
print(f"{'Fields':>6} × {'original maxItems':>18} = {'Sum':>5}  │  {'ANY':>6}  {'AUTO':>6}")
print("─" * 52)

failing_configs = [
    (1, 980),
    (8, 125),
    (20, 49),
    (50, 20),
]

for n_fields, max_items in failing_configs:
    total = n_fields * max_items
    tool = make_tool(n_fields, max_items)
    stripped = strip_max_items(tool)
    any_ok = try_call(client, stripped, "ANY")
    auto_ok = try_call(client, stripped, "AUTO")
    any_tag = "PASS ✓" if any_ok else "FAIL ✗"
    auto_tag = "PASS ✓" if auto_ok else "FAIL ✗"
    print(f"{n_fields:>6} × {max_items:>18} = {total:>5}  │  {any_tag:>6}  {auto_tag:>6}")

Same failing schemas with maxItems stripped
Fields ×  original maxItems =   Sum  │     ANY    AUTO
────────────────────────────────────────────────────
     1 ×                980 =   980  │  PASS ✓  PASS ✓
     8 ×                125 =  1000  │  PASS ✓  PASS ✓
    20 ×                 49 =   980  │  PASS ✓  PASS ✓
    50 ×                 20 =  1000  │  PASS ✓  PASS ✓


## Workarounds

1. **Strip `maxItems`/`minItems` from schemas sent to Gemini** — enforce these constraints in your own validation layer at tool invocation time
2. **Keep `maxItems` values small** — if you must include them, ensure the cumulative sum across all properties stays well under ~960

## Production Impact

This is a real-world issue for agent platforms that dynamically generate tool schemas from API specs (e.g. OpenAPI → JSON Schema). A typical lead-search tool with ~20 array fields each having `maxItems: 100` (sum=2000) will silently fail with an opaque error.